# Module 4 • Distributional Semantics and Word Embeddings

# Lesson 22 • Word2Vec — Skip-Gram and Continuous Bag-of-Words

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Estimated study time:** 120–150 minutes

---

## Scope

This lesson introduces predictive word embeddings through Word2Vec. It covers
the Continuous Bag-of-Words and Skip-Gram objectives, training-pair
construction, softmax, negative sampling, subsampling, optimization,
hyperparameters, nearest neighbors, analogy intuition, evaluation, and Arabic
considerations.

All experiments are self-contained and use NumPy rather than external model
downloads.

## Learning Objectives

After completing this lesson, the learner should be able to:

- explain the difference between count-based and predictive embeddings;
- describe the CBOW and Skip-Gram objectives;
- generate CBOW and Skip-Gram training examples;
- explain input and output embedding matrices;
- calculate logits and softmax probabilities;
- explain why full softmax is expensive;
- describe negative sampling;
- implement a small Skip-Gram model with NumPy;
- implement a small CBOW model with NumPy;
- inspect learned nearest neighbors;
- explain the effects of window size, dimension, epochs, and learning rate;
- distinguish training loss from semantic quality;
- discuss intrinsic and extrinsic evaluation;
- identify bias, frequency, and out-of-vocabulary limitations;
- explain Arabic and multilingual Word2Vec considerations.

## Table of Contents

1. From Count-Based to Predictive Embeddings
2. Word2Vec Overview
3. Training Corpus and Vocabulary
4. Context Windows
5. Skip-Gram Training Pairs
6. CBOW Training Examples
7. Input and Output Embedding Matrices
8. Softmax Objective
9. Why Full Softmax Is Expensive
10. Negative Sampling
11. Negative-Sampling Distribution
12. Skip-Gram with Negative Sampling
13. Training the Skip-Gram Model
14. Inspecting Skip-Gram Embeddings
15. CBOW with Negative Sampling
16. Training the CBOW Model
17. Comparing Skip-Gram and CBOW
18. Hyperparameters
19. Frequent-Word Subsampling
20. Nearest Neighbors
21. Analogy Intuition
22. Embedding Visualization
23. Evaluation
24. Common Failure Modes
25. Bias and Responsible Use
26. Arabic and Multilingual Considerations
27. Reproducibility and Reporting
28. Knowledge Check
29. Exercises
30. Summary and Next Lesson

# 1. From Count-Based to Predictive Embeddings

Count-based embeddings begin with observed co-occurrence statistics. Predictive
embeddings learn vector parameters by solving a prediction task.

Word2Vec learns representations by predicting:

- a target word from surrounding context; or
- surrounding context words from a target word.

In [ ]:
import math
from collections import Counter

import numpy as np
import pandas as pd

comparison = pd.DataFrame(
    [
        (
            "Count-based",
            "co-occurrence matrix",
            "PPMI + SVD",
            "explicit corpus statistics",
        ),
        (
            "Predictive",
            "context prediction",
            "Word2Vec",
            "learned optimization objective",
        ),
    ],
    columns=["Family", "Core signal", "Example", "Main property"],
)

comparison

Both approaches rely on the distributional hypothesis: words occurring in
similar contexts tend to receive similar representations.

# 2. Word2Vec Overview

Word2Vec commonly refers to two architectures:

## Continuous Bag-of-Words

Predict the target word from surrounding context words.

```text
context words → target word
```

## Skip-Gram

Predict surrounding context words from one target word.

```text
target word → context words
```

In [ ]:
architecture_table = pd.DataFrame(
    [
        (
            "CBOW",
            "context",
            "center word",
            "often faster and smoother for frequent words",
        ),
        (
            "Skip-Gram",
            "center word",
            "context words",
            "often effective for rare words and small datasets",
        ),
    ],
    columns=[
        "Architecture",
        "Input",
        "Prediction",
        "Typical behavior",
    ],
)

architecture_table

These are tendencies rather than guarantees. Results depend on corpus size,
frequency distribution, preprocessing, and hyperparameters.

# 3. Training Corpus and Vocabulary

We use a small synthetic corpus with medical and educational vocabulary.

In [ ]:
corpus = [
    "doctor treats patient in hospital",
    "nurse cares for patient in clinic",
    "physician examines patient in hospital",
    "surgeon operates in hospital",
    "teacher teaches student in school",
    "professor teaches student at university",
    "student studies lesson in school",
    "researcher works at university",
    "doctor and nurse work together",
    "teacher and professor work together",
    "hospital employs doctor and nurse",
    "university employs professor and researcher",
]

tokenized_corpus = [
    sentence.lower().split()
    for sentence in corpus
]

tokenized_corpus[:3]

In [ ]:
word_counts = Counter(
    token
    for sentence in tokenized_corpus
    for token in sentence
)

vocabulary = sorted(word_counts)
word_to_index = {
    word: index
    for index, word in enumerate(vocabulary)
}
index_to_word = {
    index: word
    for word, index in word_to_index.items()
}

print("Vocabulary size:", len(vocabulary))
print("Vocabulary:", vocabulary)

Word2Vec usually discards words below a minimum-frequency threshold. This
controls memory use and reduces noise, but it can remove rare and important
terms.

# 4. Context Windows

A window determines which words count as context.

Example with window size 2:

```text
doctor treats patient in hospital
       ↑ target
```

If `patient` is the target, nearby context words may be:

```text
doctor, treats, in, hospital
```

In [ ]:
def context_indices(
    tokens: list[str],
    center_index: int,
    window_size: int,
) -> list[int]:
    left = max(0, center_index - window_size)
    right = min(len(tokens), center_index + window_size + 1)

    return [
        index
        for index in range(left, right)
        if index != center_index
    ]


example_tokens = "doctor treats patient in hospital".split()
center_index = example_tokens.index("patient")

context_words = [
    example_tokens[index]
    for index in context_indices(
        example_tokens,
        center_index,
        window_size=2,
    )
]

context_words

Smaller windows often emphasize syntactic and functional similarity. Larger
windows often emphasize broader topical relatedness.

# 5. Skip-Gram Training Pairs

Skip-Gram creates one pair for each center-context relation.

Example:

```text
center = patient
pairs:
patient → doctor
patient → treats
patient → in
patient → hospital
```

In [ ]:
def build_skipgram_pairs(
    tokenized_sentences: list[list[str]],
    mapping: dict[str, int],
    window_size: int,
) -> list[tuple[int, int]]:
    pairs = []

    for sentence in tokenized_sentences:
        for center_position, center_word in enumerate(sentence):
            center_id = mapping[center_word]

            for context_position in context_indices(
                sentence,
                center_position,
                window_size,
            ):
                context_word = sentence[context_position]
                context_id = mapping[context_word]
                pairs.append((center_id, context_id))

    return pairs


skipgram_pairs = build_skipgram_pairs(
    tokenized_corpus,
    word_to_index,
    window_size=2,
)

print("Number of pairs:", len(skipgram_pairs))
print(
    [
        (
            index_to_word[center],
            index_to_word[context],
        )
        for center, context in skipgram_pairs[:12]
    ]
)

One sentence generates many Skip-Gram examples, which can increase training
cost.

# 6. CBOW Training Examples

CBOW combines context words to predict the center word.

In [ ]:
def build_cbow_examples(
    tokenized_sentences: list[list[str]],
    mapping: dict[str, int],
    window_size: int,
) -> list[tuple[list[int], int]]:
    examples = []

    for sentence in tokenized_sentences:
        for center_position, center_word in enumerate(sentence):
            context_positions = context_indices(
                sentence,
                center_position,
                window_size,
            )

            context_ids = [
                mapping[sentence[position]]
                for position in context_positions
            ]

            if not context_ids:
                continue

            examples.append(
                (
                    context_ids,
                    mapping[center_word],
                )
            )

    return examples


cbow_examples = build_cbow_examples(
    tokenized_corpus,
    word_to_index,
    window_size=2,
)

print("Number of CBOW examples:", len(cbow_examples))

context_ids, target_id = cbow_examples[5]

print(
    "Context:",
    [index_to_word[index] for index in context_ids],
)
print("Target:", index_to_word[target_id])

CBOW usually averages or sums the context embeddings before predicting the
target.

# 7. Input and Output Embedding Matrices

Word2Vec maintains two parameter matrices:

- input embeddings;
- output embeddings.

For vocabulary size \(V\) and embedding dimension \(D\):

```text
input matrix:  V × D
output matrix: V × D
```

In [ ]:
import numpy as np

random_generator = np.random.default_rng(42)

vocabulary_size = len(vocabulary)
embedding_dimension = 12

input_embeddings = random_generator.normal(
    loc=0.0,
    scale=0.1,
    size=(vocabulary_size, embedding_dimension),
)

output_embeddings = np.zeros(
    (vocabulary_size, embedding_dimension),
    dtype=float,
)

print("Input shape:", input_embeddings.shape)
print("Output shape:", output_embeddings.shape)

The input matrix is commonly used as the final embedding table, although some
implementations combine input and output vectors.

# 8. Softmax Objective

Given a center-word vector \(v_w\) and output vector \(u_c\), the model
calculates a score:

\[
score(w,c) = u_c^	op v_w
\]

Full softmax converts scores over the complete vocabulary into probabilities.

In [ ]:
def stable_softmax(values: np.ndarray) -> np.ndarray:
    shifted = values - values.max()
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum()


center_word = "doctor"
center_id = word_to_index[center_word]
center_vector = input_embeddings[center_id]

logits = output_embeddings @ center_vector
probabilities = stable_softmax(logits)

print("Probability sum:", probabilities.sum())
print("First five probabilities:", probabilities[:5])

Training increases probability for observed center-context pairs and decreases
probability for alternatives.

# 9. Why Full Softmax Is Expensive

Full softmax calculates scores for every vocabulary item for every training
example.

With a vocabulary of millions of words, this becomes expensive in:

- computation;
- memory bandwidth;
- gradient updates.

Common alternatives include:

- hierarchical softmax;
- negative sampling;
- sampled softmax.

# 10. Negative Sampling

Negative sampling converts one multiclass prediction into several binary
decisions.

For an observed pair:

```text
doctor → patient
```

The model learns:

```text
doctor + patient   = positive
doctor + school    = negative
doctor + lesson    = negative
```

The positive pair should receive a high sigmoid score. Negative pairs should
receive low sigmoid scores.

In [ ]:
def sigmoid(value):
    clipped = np.clip(value, -20, 20)
    return 1.0 / (1.0 + np.exp(-clipped))


for value in [-4, -1, 0, 1, 4]:
    print(value, "->", round(float(sigmoid(value)), 4))

# 11. Negative-Sampling Distribution

Negative words are not usually sampled uniformly.

A common distribution is proportional to:

\[
frequency(word)^{0.75}
\]

The exponent reduces the dominance of extremely frequent words while still
sampling them more often than rare words.

In [ ]:
frequencies = np.array(
    [word_counts[word] for word in vocabulary],
    dtype=float,
)

negative_distribution = frequencies ** 0.75
negative_distribution /= negative_distribution.sum()

sampling_frame = pd.DataFrame(
    {
        "word": vocabulary,
        "frequency": frequencies.astype(int),
        "negative_probability": negative_distribution,
    }
).sort_values(
    "negative_probability",
    ascending=False,
)

sampling_frame.head(10)

In [ ]:
def sample_negative_ids(
    count: int,
    positive_id: int,
    distribution: np.ndarray,
    generator: np.random.Generator,
) -> list[int]:
    negatives = []

    while len(negatives) < count:
        sampled_id = int(
            generator.choice(
                len(distribution),
                p=distribution,
            )
        )

        if sampled_id != positive_id:
            negatives.append(sampled_id)

    return negatives


sample_negative_ids(
    count=5,
    positive_id=word_to_index["patient"],
    distribution=negative_distribution,
    generator=random_generator,
)

# 12. Skip-Gram with Negative Sampling

For each positive center-context pair:

1. calculate the positive score;
2. update vectors to increase that score;
3. sample negative words;
4. update vectors to decrease negative scores.

In [ ]:
def train_skipgram_negative_sampling(
    pairs: list[tuple[int, int]],
    vocab_size: int,
    dimension: int,
    negative_distribution: np.ndarray,
    epochs: int = 80,
    learning_rate: float = 0.03,
    negatives_per_positive: int = 4,
    seed: int = 42,
) -> tuple[np.ndarray, np.ndarray, list[float]]:
    generator = np.random.default_rng(seed)

    input_matrix = generator.normal(
        0.0,
        0.1,
        size=(vocab_size, dimension),
    )

    output_matrix = generator.normal(
        0.0,
        0.1,
        size=(vocab_size, dimension),
    )

    losses = []
    pair_array = np.array(pairs, dtype=int)

    for epoch in range(epochs):
        generator.shuffle(pair_array)
        epoch_loss = 0.0

        current_rate = learning_rate * (
            1.0 - 0.7 * epoch / max(epochs - 1, 1)
        )

        for center_id, positive_id in pair_array:
            center_vector = input_matrix[center_id].copy()
            positive_vector = output_matrix[positive_id].copy()

            positive_score = float(
                sigmoid(
                    np.dot(center_vector, positive_vector)
                )
            )

            positive_error = positive_score - 1.0
            epoch_loss -= math.log(
                max(positive_score, 1e-12)
            )

            center_gradient = (
                positive_error * positive_vector
            )

            output_matrix[positive_id] -= (
                current_rate
                * positive_error
                * center_vector
            )

            negative_ids = sample_negative_ids(
                negatives_per_positive,
                positive_id,
                negative_distribution,
                generator,
            )

            for negative_id in negative_ids:
                negative_vector = output_matrix[
                    negative_id
                ].copy()

                negative_score = float(
                    sigmoid(
                        np.dot(
                            center_vector,
                            negative_vector,
                        )
                    )
                )

                negative_error = negative_score
                epoch_loss -= math.log(
                    max(
                        1.0 - negative_score,
                        1e-12,
                    )
                )

                center_gradient += (
                    negative_error
                    * negative_vector
                )

                output_matrix[negative_id] -= (
                    current_rate
                    * negative_error
                    * center_vector
                )

            input_matrix[center_id] -= (
                current_rate * center_gradient
            )

        losses.append(
            epoch_loss / len(pair_array)
        )

    return input_matrix, output_matrix, losses

This educational implementation prioritizes clarity over computational
efficiency.

# 13. Training the Skip-Gram Model

In [ ]:
skipgram_input, skipgram_output, skipgram_losses = (
    train_skipgram_negative_sampling(
        pairs=skipgram_pairs,
        vocab_size=vocabulary_size,
        dimension=12,
        negative_distribution=negative_distribution,
        epochs=100,
        learning_rate=0.04,
        negatives_per_positive=5,
        seed=42,
    )
)

print("Initial loss:", round(skipgram_losses[0], 4))
print("Final loss:", round(skipgram_losses[-1], 4))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(skipgram_losses)
plt.title("Skip-Gram Negative-Sampling Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Average loss")
plt.tight_layout()
plt.show()

Decreasing training loss indicates progress on the prediction objective, not
necessarily high-quality semantic embeddings.

# 14. Inspecting Skip-Gram Embeddings

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def nearest_neighbors(
    target_word: str,
    embeddings: np.ndarray,
    top_k: int = 6,
) -> pd.DataFrame:
    target_id = word_to_index[target_word]
    target_vector = embeddings[target_id].reshape(1, -1)

    scores = cosine_similarity(
        target_vector,
        embeddings,
    ).ravel()

    rows = []

    for word_id, score in enumerate(scores):
        word = index_to_word[word_id]

        if word == target_word:
            continue

        rows.append(
            {
                "word": word,
                "similarity": float(score),
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(
            ["similarity", "word"],
            ascending=[False, True],
        )
        .head(top_k)
        .reset_index(drop=True)
    )


nearest_neighbors(
    "doctor",
    skipgram_input,
)

In [ ]:
for target in [
    "doctor",
    "nurse",
    "teacher",
    "professor",
    "hospital",
    "university",
]:
    print(target)
    display(
        nearest_neighbors(
            target,
            skipgram_input,
            top_k=4,
        )
    )

The corpus is small, so neighbor quality is unstable and should be interpreted
as a demonstration rather than a benchmark.

# 15. CBOW with Negative Sampling

CBOW averages context vectors and uses the result to predict the center word.

In [ ]:
def train_cbow_negative_sampling(
    examples: list[tuple[list[int], int]],
    vocab_size: int,
    dimension: int,
    negative_distribution: np.ndarray,
    epochs: int = 80,
    learning_rate: float = 0.03,
    negatives_per_positive: int = 4,
    seed: int = 42,
) -> tuple[np.ndarray, np.ndarray, list[float]]:
    generator = np.random.default_rng(seed)

    input_matrix = generator.normal(
        0.0,
        0.1,
        size=(vocab_size, dimension),
    )

    output_matrix = generator.normal(
        0.0,
        0.1,
        size=(vocab_size, dimension),
    )

    losses = []

    for epoch in range(epochs):
        order = generator.permutation(len(examples))
        epoch_loss = 0.0

        current_rate = learning_rate * (
            1.0 - 0.7 * epoch / max(epochs - 1, 1)
        )

        for example_index in order:
            context_ids, positive_id = examples[
                example_index
            ]

            context_matrix = input_matrix[
                context_ids
            ]

            hidden_vector = context_matrix.mean(axis=0)
            positive_vector = output_matrix[
                positive_id
            ].copy()

            positive_score = float(
                sigmoid(
                    np.dot(
                        hidden_vector,
                        positive_vector,
                    )
                )
            )

            positive_error = positive_score - 1.0
            epoch_loss -= math.log(
                max(positive_score, 1e-12)
            )

            hidden_gradient = (
                positive_error
                * positive_vector
            )

            output_matrix[positive_id] -= (
                current_rate
                * positive_error
                * hidden_vector
            )

            negative_ids = sample_negative_ids(
                negatives_per_positive,
                positive_id,
                negative_distribution,
                generator,
            )

            for negative_id in negative_ids:
                negative_vector = output_matrix[
                    negative_id
                ].copy()

                negative_score = float(
                    sigmoid(
                        np.dot(
                            hidden_vector,
                            negative_vector,
                        )
                    )
                )

                negative_error = negative_score
                epoch_loss -= math.log(
                    max(
                        1.0 - negative_score,
                        1e-12,
                    )
                )

                hidden_gradient += (
                    negative_error
                    * negative_vector
                )

                output_matrix[negative_id] -= (
                    current_rate
                    * negative_error
                    * hidden_vector
                )

            context_gradient = (
                hidden_gradient
                / len(context_ids)
            )

            for context_id in context_ids:
                input_matrix[context_id] -= (
                    current_rate
                    * context_gradient
                )

        losses.append(
            epoch_loss / len(examples)
        )

    return input_matrix, output_matrix, losses

# 16. Training the CBOW Model

In [ ]:
cbow_input, cbow_output, cbow_losses = (
    train_cbow_negative_sampling(
        examples=cbow_examples,
        vocab_size=vocabulary_size,
        dimension=12,
        negative_distribution=negative_distribution,
        epochs=100,
        learning_rate=0.04,
        negatives_per_positive=5,
        seed=42,
    )
)

print("Initial loss:", round(cbow_losses[0], 4))
print("Final loss:", round(cbow_losses[-1], 4))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(cbow_losses)
plt.title("CBOW Negative-Sampling Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Average loss")
plt.tight_layout()
plt.show()

In [ ]:
nearest_neighbors(
    "teacher",
    cbow_input,
)

# 17. Comparing Skip-Gram and CBOW

The same target word may receive different neighbors under each objective.

In [ ]:
comparison_rows = []

for target in [
    "doctor",
    "nurse",
    "teacher",
    "professor",
]:
    skipgram_neighbors = nearest_neighbors(
        target,
        skipgram_input,
        top_k=3,
    )["word"].tolist()

    cbow_neighbors = nearest_neighbors(
        target,
        cbow_input,
        top_k=3,
    )["word"].tolist()

    comparison_rows.append(
        {
            "target": target,
            "skipgram_neighbors": ", ".join(
                skipgram_neighbors
            ),
            "cbow_neighbors": ", ".join(
                cbow_neighbors
            ),
        }
    )

pd.DataFrame(comparison_rows)

Differences may result from the objective, optimization noise, small corpus
size, or hyperparameters.

# 18. Hyperparameters

Important Word2Vec hyperparameters include:

- architecture: CBOW or Skip-Gram;
- embedding dimension;
- context window;
- minimum word frequency;
- number of negative samples;
- learning rate;
- number of epochs;
- subsampling threshold;
- random seed.

In [ ]:
hyperparameter_table = pd.DataFrame(
    [
        ("Window size", "local syntax versus broad topic"),
        ("Dimension", "capacity, memory, overfitting"),
        ("Negative samples", "training cost and contrast"),
        ("Learning rate", "speed and stability"),
        ("Epochs", "undertraining versus overfitting"),
        ("Minimum frequency", "coverage versus noise"),
    ],
    columns=["Hyperparameter", "Primary effect"],
)

hyperparameter_table

Hyperparameters should be selected using validation tasks rather than training
loss alone.

# 19. Frequent-Word Subsampling

Very frequent words generate many training pairs but may provide limited
semantic information.

Word2Vec can randomly discard frequent tokens with probability related to their
corpus frequency.

In [ ]:
total_token_count = sum(word_counts.values())

token_probabilities = {
    word: count / total_token_count
    for word, count in word_counts.items()
}

subsampling_threshold = 1e-2

keep_probabilities = {}

for word, probability in token_probabilities.items():
    keep_probability = min(
        1.0,
        (
            math.sqrt(
                subsampling_threshold
                / probability
            )
            + 1
        )
        * (
            subsampling_threshold
            / probability
        ),
    )

    keep_probabilities[word] = keep_probability

subsampling_frame = pd.DataFrame(
    [
        (
            word,
            word_counts[word],
            token_probabilities[word],
            keep_probabilities[word],
        )
        for word in vocabulary
    ],
    columns=[
        "word",
        "frequency",
        "corpus_probability",
        "keep_probability",
    ],
).sort_values("keep_probability")

subsampling_frame.head(10)

Subsampling reduces training cost and prevents frequent function words from
dominating the objective.

# 20. Nearest Neighbors

Nearest-neighbor evaluation should inspect:

- semantic similarity;
- topical relatedness;
- syntactic similarity;
- frequency artifacts;
- rare-word instability.

In [ ]:
qualitative_audit = pd.DataFrame(
    [
        ("doctor", "physician", "semantic similarity"),
        ("doctor", "hospital", "topical association"),
        ("teacher", "professor", "role similarity"),
        ("school", "university", "institutional relation"),
    ],
    columns=[
        "Target",
        "Possible neighbor",
        "Relation type",
    ],
)

qualitative_audit

Neighbor quality should be compared across seeds and corpus samples.

# 21. Analogy Intuition

Word2Vec embeddings may support approximate vector relations:

```text
professor - university + hospital ≈ doctor
```

This is not guaranteed and is highly corpus-dependent.

In [ ]:
def analogy(
    positive_a: str,
    negative: str,
    positive_b: str,
    embeddings: np.ndarray,
    top_k: int = 5,
) -> pd.DataFrame:
    required = {
        positive_a,
        negative,
        positive_b,
    }

    target_vector = (
        embeddings[word_to_index[positive_a]]
        - embeddings[word_to_index[negative]]
        + embeddings[word_to_index[positive_b]]
    ).reshape(1, -1)

    scores = cosine_similarity(
        target_vector,
        embeddings,
    ).ravel()

    rows = []

    for word_id, score in enumerate(scores):
        word = index_to_word[word_id]

        if word in required:
            continue

        rows.append(
            {
                "word": word,
                "similarity": float(score),
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(
            ["similarity", "word"],
            ascending=[False, True],
        )
        .head(top_k)
        .reset_index(drop=True)
    )


analogy(
    "professor",
    "university",
    "hospital",
    skipgram_input,
)

The toy corpus is not large enough to evaluate analogies reliably.

# 22. Embedding Visualization

Two-dimensional projections provide a qualitative view but distort the
original geometry.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(
    n_components=2,
    random_state=42,
)

skipgram_2d = pca.fit_transform(skipgram_input)

plot_words = [
    "doctor",
    "nurse",
    "physician",
    "surgeon",
    "teacher",
    "professor",
    "student",
    "hospital",
    "clinic",
    "school",
    "university",
]

plot_ids = [
    word_to_index[word]
    for word in plot_words
]

plt.figure(figsize=(9, 6))
plt.scatter(
    skipgram_2d[plot_ids, 0],
    skipgram_2d[plot_ids, 1],
)

for word, word_id in zip(plot_words, plot_ids):
    plt.text(
        skipgram_2d[word_id, 0],
        skipgram_2d[word_id, 1],
        word,
    )

plt.title("Skip-Gram Embeddings Projected with PCA")
plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.tight_layout()
plt.show()

Proximity in the plot is exploratory evidence, not a formal evaluation metric.

# 23. Evaluation

## Intrinsic evaluation

- similarity correlation;
- analogy accuracy;
- nearest-neighbor inspection;
- clustering.

## Extrinsic evaluation

- text classification;
- Named Entity Recognition;
- retrieval;
- machine translation;
- semantic similarity.

In [ ]:
evaluation_table = pd.DataFrame(
    [
        ("Intrinsic", "word similarity", "direct vector quality"),
        ("Intrinsic", "analogies", "relational structure"),
        ("Extrinsic", "classification", "downstream usefulness"),
        ("Extrinsic", "retrieval", "ranking quality"),
    ],
    columns=["Evaluation family", "Task", "Measures"],
)

evaluation_table

Training loss should not be reported as the sole measure of embedding quality.

# 24. Common Failure Modes

- corpus too small;
- vocabulary too sparse;
- rare words undertrained;
- frequent words dominating;
- unstable neighbors across seeds;
- poor tokenization;
- domain mismatch;
- one vector mixing several senses;
- analogies over-interpreted;
- training loss mistaken for semantic quality.

In [ ]:
failure_modes = pd.DataFrame(
    [
        ("Rare-word instability", "increase data or use subwords"),
        ("Frequent-word dominance", "subsample frequent tokens"),
        ("Domain mismatch", "train or adapt on target-domain text"),
        ("Polysemy", "use contextual embeddings"),
        ("OOV words", "use subword or character-aware models"),
        ("Unstable neighbors", "compare seeds and confidence"),
    ],
    columns=["Failure", "Possible response"],
)

failure_modes

# 25. Bias and Responsible Use

Word2Vec learns associations present in its training corpus. These may include
harmful stereotypes and historical inequalities.

Evaluation should examine:

- demographic names;
- occupations;
- geographic terms;
- religious terms;
- language varieties;
- downstream disparities.

Debiasing one embedding direction does not guarantee removal of all harmful
associations.

# 26. Arabic and Multilingual Considerations

Arabic Word2Vec training is affected by:

- clitics;
- rich morphology;
- diacritics;
- Alef and Ya variants;
- Modern Standard Arabic and dialects;
- spelling variation;
- code-switching;
- Arabizi.

In [ ]:
arabic_corpus = [
    "الطبيب يعالج المريض في المستشفى",
    "الممرضة ترعى المريض في العيادة",
    "الأستاذ يدرس الطالب في الجامعة",
    "المعلم يدرس الطالب في المدرسة",
    "الطبيب والممرضة يعملان معا",
    "المعلم والأستاذ يعملان معا",
]

arabic_tokenized = [
    sentence.split()
    for sentence in arabic_corpus
]

arabic_counts = Counter(
    token
    for sentence in arabic_tokenized
    for token in sentence
)

arabic_counts

Word-level Arabic models may assign separate vectors to:

```text
كتاب
الكتاب
والكتاب
بالكتاب
```

Subword models can share information across these forms.

In [ ]:
arabic_forms = pd.DataFrame(
    [
        ("كتاب", "bare noun"),
        ("الكتاب", "definite form"),
        ("والكتاب", "conjunction + definite form"),
        ("بالكتاب", "preposition + definite form"),
    ],
    columns=["Form", "Description"],
)

arabic_forms

Multilingual embeddings may be trained jointly or aligned after monolingual
training. Alignment quality depends on shared anchors and comparable domains.

# 27. Reproducibility and Reporting

Report:

- corpus source and size;
- preprocessing;
- vocabulary size;
- minimum frequency;
- architecture;
- context window;
- embedding dimension;
- negative samples;
- subsampling threshold;
- learning rate schedule;
- epochs;
- random seed;
- evaluation tasks.

In [ ]:
import platform

metadata = pd.Series(
    {
        "architecture": "Skip-Gram and CBOW",
        "objective": "Negative sampling",
        "corpus_sentences": len(corpus),
        "vocabulary_size": vocabulary_size,
        "window_size": 2,
        "embedding_dimension": 12,
        "negative_samples": 5,
        "epochs": 100,
        "random_seed": 42,
        "python_version": platform.python_version(),
        "numpy_version": np.__version__,
    },
    name="Word2Vec experiment",
)

metadata

Reproducibility requires the exact corpus and vocabulary—not only the final
vectors.

# 28. Knowledge Check

1. How do predictive embeddings differ from count-based embeddings?
2. What does CBOW predict?
3. What does Skip-Gram predict?
4. How are Skip-Gram training pairs generated?
5. How are CBOW context vectors combined?
6. Why does Word2Vec use input and output matrices?
7. What does full softmax calculate?
8. Why is full softmax expensive?
9. What is negative sampling?
10. Why is the negative distribution raised to the 0.75 power?
11. Why can training loss decrease while semantic quality remains poor?
12. What does frequent-word subsampling achieve?
13. How do window size and embedding dimension affect results?
14. Why do static Word2Vec embeddings struggle with polysemy?
15. Which Arabic characteristics affect Word2Vec training?

# 29. Exercises

## Exercise 1 — Training Pairs

Generate Skip-Gram and CBOW examples for different window sizes.

## Exercise 2 — Full Softmax

Implement a small full-softmax Skip-Gram model and compare its cost with
negative sampling.

## Exercise 3 — Negative Sampling

Compare uniform negative sampling with frequency-powered sampling.

## Exercise 4 — Hyperparameters

Compare embedding dimensions 8, 16, and 32.

## Exercise 5 — Architectures

Compare Skip-Gram and CBOW nearest neighbors on the same corpus.

## Exercise 6 — Subsampling

Add frequent-word subsampling before pair construction.

## Exercise 7 — Evaluation

Build a small similarity dataset and calculate Spearman correlation.

## Exercise 8 — Arabic Word2Vec

Compare raw Arabic tokens with lightly normalized or segmented tokens.

## Challenge Exercises

1. Vectorize the training loop for greater efficiency.
2. Implement hierarchical softmax.
3. Add dynamic context windows.
4. Add phrase detection before Word2Vec training.
5. Compare input vectors, output vectors, and their average.

# 30. Summary and Next Lesson

In this lesson:

- Word2Vec was introduced as a predictive embedding method;
- CBOW predicted center words from context;
- Skip-Gram predicted context words from centers;
- context windows generated training examples;
- input and output embedding matrices represented distinct roles;
- full softmax was shown to be computationally expensive;
- negative sampling converted prediction into binary discrimination;
- frequency-powered sampling balanced common and rare negatives;
- NumPy implementations trained Skip-Gram and CBOW models;
- nearest neighbors and visualizations supported qualitative inspection;
- hyperparameters and subsampling affected training behavior;
- analogy arithmetic was treated as approximate;
- intrinsic and extrinsic evaluation were distinguished;
- corpus bias, polysemy, and OOV limitations were identified;
- Arabic Word2Vec required morphology- and script-aware preprocessing.

## Next Lesson

**Lesson 23: GloVe, FastText, and Subword-Aware Embeddings** compares global
matrix-factorization embeddings with character n-gram methods and explores
robust representations for rare and morphologically complex words.

# References

- Mikolov, T., Chen, K., Corrado, G., & Dean, J. *Efficient Estimation of Word Representations in Vector Space*.
- Mikolov, T. et al. *Distributed Representations of Words and Phrases and their Compositionality*.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.
- Word2Vec, negative sampling, and embedding-evaluation literature.